In [1]:
import time
import random
import threading
from collections import deque

class WearableDevice:
    """
    Simulates a wearable IoT device (e.g., a smartwatch) that collects sensor data.
    """
    def __init__(self, device_id, sampling_rate=1):
        """
        Initializes the wearable device.

        Args:
            device_id (str): Unique identifier for the device.
            sampling_rate (int): The rate at which data is collected (in Hz).
        """
        self.device_id = device_id
        self.sampling_rate = sampling_rate  # Hz (samples per second)
        self.heart_rate = 70  # Initial heart rate
        self.oxygen_level = 98  # Initial oxygen level (%)
        self.steps = 0  # Initial step count
        self.battery_level = 100  # Initial battery level (%)
        self.running = True
        self.data_queue = deque(maxlen=100)  # Store last 100 data points

    def get_sensor_data(self):
        """
        Simulates getting sensor data.  In a real application, this would involve
        reading data from actual sensors.  Here, we add a small random variation
        to simulate real-world fluctuations.  We also simulate battery drain.
        """
        if not self.running:
            return None

        # Simulate heart rate variation (resting: 60-100, exercise: up to 180)
        self.heart_rate += random.randint(-5, 5)
        self.heart_rate = max(40, min(self.heart_rate, 190))  # Keep within bounds

        # Simulate oxygen level variation (normal: 95-100%, low: below 90%)
        self.oxygen_level += random.uniform(-0.2, 0.2)
        self.oxygen_level = max(80, min(self.oxygen_level, 100))

        # Simulate step count (more steps if heart rate is elevated)
        if self.heart_rate > 100:
            self.steps += random.randint(5, 15)
        else:
            self.steps += random.randint(1, 5)

        # Simulate battery drain (simple linear drain, drain faster if HR is high)
        if self.heart_rate > 120:
            self.battery_level -= 0.5  # Increased drain during exercise
        else:
            self.battery_level -= 0.1
        self.battery_level = max(0, self.battery_level)  # Ensure not negative

        #package the data
        data = {
            "heart_rate": self.heart_rate,
            "oxygen_level": self.oxygen_level,
            "steps": self.steps,
            "battery_level": self.battery_level,
            "timestamp": time.time()
        }
        self.data_queue.append(data) #store the data

        return data

    def run(self):
        """Simulates the device running and generating data."""
        self.running = True

    def stop(self):
        """Simulates the device being stopped."""
        self.running = False

    def get_data_history(self):
        """Returns the data history."""
        return list(self.data_queue)


    def __repr__(self):
        return (f"WearableDevice(id={self.device_id}, heart_rate={self.heart_rate}, "
                f"oxygen_level={self.oxygen_level:.1f}, steps={self.steps}, "
                f"battery_level={self.battery_level:.1f}, running={self.running})")



class DataProcessor:
    """
    Simulates processing data from a wearable device.  This could involve
    filtering, analysis, or sending data to a cloud platform.
    """
    def __init__(self, device_id):
        """
        Initializes the data processor.

        Args:
            device_id (str): The ID of the wearable device to process data from.
        """
        self.device_id = device_id
        self.processed_data = []
        self.running = True

    def process_data(self, data):
        """
        Processes the sensor data.  This is a simplified example; in a real
        application, you might perform more complex analysis.

        Args:
            data (dict): A dictionary containing the sensor data.
        """
        if not self.running or data is None:
            return

        # Example: Calculate a simple health score (higher is better)
        health_score = (data["oxygen_level"] + (190 - data["heart_rate"]) / 2) * (data["battery_level"] / 100)
        processed_data = {
            "device_id": self.device_id,
            "timestamp": data['timestamp'],
            "heart_rate": data["heart_rate"],
            "oxygen_level": data["oxygen_level"],
            "steps": data["steps"],
            "battery_level": data["battery_level"],
            "health_score": health_score
        }
        self.processed_data.append(processed_data)
        print(f"Processed data from {self.device_id}: {processed_data}")

    def run(self):
        """Sets the processor to running."""
        self.running = True

    def stop(self):
        """Sets the processor to stop."""
        self.running = False

    def get_processed_data(self):
        """Returns all processed data"""
        return self.processed_data

    def get_recent_processed_data(self, num_last_entries=10):
        """Returns the last n processed data entries"""
        return self.processed_data[-num_last_entries:]

def simulate_data_flow(device, processor):
    """
    Simulates the flow of data from a wearable device to a data processor.
    Runs in a separate thread to mimic real-time data transmission.

    Args:
        device (WearableDevice): The wearable device to simulate.
        processor (DataProcessor): The data processor to send data to.
    """
    while True:
        sensor_data = device.get_sensor_data()  # Get data from the device
        processor.process_data(sensor_data)  # Process the data
        time.sleep(device.sampling_rate)  # Wait for the next sample

if __name__ == "__main__":
    # Create a wearable device and a data processor
    wearable_device = WearableDevice("Device001", sampling_rate=1)  # Collect data at 1Hz
    data_processor = DataProcessor("Device001")

    # Start the data flow simulation in a separate thread
    data_thread = threading.Thread(target=simulate_data_flow, args=(wearable_device, data_processor,))
    data_thread.daemon = True  # Allow the main thread to exit
    data_thread.start()

    # Simulate the device running for a while
    print(f"Initial Device State: {wearable_device}")
    time.sleep(10)  # Simulate data collection for 10 seconds

    # Simulate the device being stopped
    print("Stopping data collection...")
    wearable_device.stop()
    data_processor.stop()
    time.sleep(2)  # Wait for any remaining data to be processed

    # Print the final state of the device
    print(f"Final Device State: {wearable_device}")
    print(f"All processed data: {data_processor.get_processed_data()}")
    print(f"Last 5 processed data entries: {data_processor.get_recent_processed_data(5)}")

    print("Exiting...")


Processed data from Device001: {'device_id': 'Device001', 'timestamp': 1745895232.926545, 'heart_rate': 74, 'oxygen_level': 97.80347475821394, 'steps': 2, 'battery_level': 99.9, 'health_score': 155.64767128345576}
Initial Device State: WearableDevice(id=Device001, heart_rate=74, oxygen_level=97.8, steps=2, battery_level=99.9, running=True)
Processed data from Device001: {'device_id': 'Device001', 'timestamp': 1745895233.9285207, 'heart_rate': 70, 'oxygen_level': 97.9215408062639, 'steps': 6, 'battery_level': 99.80000000000001, 'health_score': 157.6056977246514}
Processed data from Device001: {'device_id': 'Device001', 'timestamp': 1745895234.929181, 'heart_rate': 72, 'oxygen_level': 98.11155305790807, 'steps': 11, 'battery_level': 99.70000000000002, 'health_score': 156.64021839873436}
Processed data from Device001: {'device_id': 'Device001', 'timestamp': 1745895235.9296534, 'heart_rate': 71, 'oxygen_level': 98.3053943910743, 'steps': 15, 'battery_level': 99.60000000000002, 'health_scor